In [1]:
import numpy as np
from pathlib import Path

output_dir = Path("../data/processed_data/embeddings")

train_embeddings = np.load(output_dir / "train_embeddings.npy",allow_pickle=True)
dev_embeddings = np.load(output_dir / "dev_embeddings.npy",allow_pickle=True)

train_labels = np.load(output_dir / "train_labels.npy",allow_pickle=True)
dev_labels = np.load(output_dir / "dev_labels.npy",allow_pickle=True)

In [3]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(
    max_iter=2000,
    random_state=14
)

classifier.fit(train_embeddings, train_labels)
print("Classifier trained!")

Classifier trained!


In [4]:
# Predict labels for the development set
dev_predictions = classifier.predict(dev_embeddings)

FROZEN ROBERTA + LR

In [13]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report
)

# Calculate evaluation metrics
accuracy = accuracy_score(dev_labels, dev_predictions)

macro_f1 = f1_score(
    dev_labels,
    dev_predictions,
    average="macro"
)

weighted_f1 = f1_score(
    dev_labels,
    dev_predictions,
    average="weighted"
)

balanced_acc = balanced_accuracy_score(
    dev_labels,
    dev_predictions
)

mcc = matthews_corrcoef(
    dev_labels,
    dev_predictions
)

print("Accuracy:", round(accuracy, 4))
print("Macro-F1:", round(macro_f1, 4))
print("Weighted-F1:", round(weighted_f1, 4))
print("Balanced Accuracy:", round(balanced_acc, 4))
print("MCC:", round(mcc, 4))

print("\nClassification Report:")
print(classification_report(dev_labels, dev_predictions))

NameError: name 'dev_predictions' is not defined

FINE TUNINIG TESTING 

In [3]:
import pandas as pd

dev_df = pd.read_csv("../data/processed_data/dev.csv")

label2id = {
    "not depression": 0,
    "moderate": 1,
    "severe": 2
}

id2label = {v: k for k, v in label2id.items()}

dev_texts = dev_df["Text_data"].astype(str).tolist()
dev_labels_text = dev_df["Label"].tolist()
dev_labels = np.array([label2id[label] for label in dev_labels_text])

print("Development samples:", len(dev_texts))
print("Label distribution:")
print(dev_df["Label"].value_counts())

Development samples: 1373
Label distribution:
Label
moderate          703
not depression    529
severe            141
Name: count, dtype: int64


In [4]:
dev_embedding_labels = np.load(
    output_dir / "dev_labels.npy",
    allow_pickle=True
)

print("CSV labels:", np.unique(dev_labels, return_counts=True))
print("Embedding labels:", np.unique(dev_embedding_labels, return_counts=True))
print("Labels identical:", np.array_equal(dev_labels, dev_embedding_labels))

CSV labels: (array([0, 1, 2]), array([529, 703, 141]))
Embedding labels: (array(['moderate', 'not depression', 'severe'], dtype=object), array([703, 529, 141]))
Labels identical: False


In [5]:
dev_embedding_labels_encoded = np.array([
    label2id[label] for label in dev_embedding_labels
])

print(
    "Embedding labels match CSV labels:",
    np.array_equal(dev_labels, dev_embedding_labels_encoded)
)

Embedding labels match CSV labels: True


In [6]:
from pathlib import Path

roberta_dir = Path("roberta_finetuned_results")
distilbert_dir = Path("distilbert_results")

print("RoBERTa checkpoints:")
print(sorted(str(p) for p in roberta_dir.glob("checkpoint-*")))

print("\nDistilBERT checkpoints:")
print(sorted(str(p) for p in distilbert_dir.glob("checkpoint-*")))

RoBERTa checkpoints:
['roberta_finetuned_results/checkpoint-1372', 'roberta_finetuned_results/checkpoint-2058', 'roberta_finetuned_results/checkpoint-686']

DistilBERT checkpoints:
['distilbert_results/checkpoint-1372', 'distilbert_results/checkpoint-2058']


In [27]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

roberta_checkpoint = Path("roberta_finetuned_results/checkpoint-2058")  

roberta_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta_checkpoint
)

roberta_model.eval()

print("RoBERTa loaded from:", roberta_checkpoint)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RoBERTa loaded from: roberta_finetuned_results/checkpoint-2058


In [28]:
MAX_LENGTH = 256  # Change this if your RoBERTa training used a different value

roberta_encodings = roberta_tokenizer(
    dev_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

print("Number of samples:", roberta_encodings["input_ids"].shape[0])
print("Tokenized shape:", roberta_encodings["input_ids"].shape)

Number of samples: 1373
Tokenized shape: torch.Size([1373, 256])


In [10]:
import torch

In [29]:
from torch.utils.data import DataLoader, TensorDataset

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

roberta_model.to(device)

roberta_dataset = TensorDataset(
    roberta_encodings["input_ids"],
    roberta_encodings["attention_mask"]
)

roberta_loader = DataLoader(
    roberta_dataset,
    batch_size=8,
    shuffle=False
)

roberta_predictions = []

roberta_model.eval()

with torch.no_grad():
    for input_ids, attention_mask in roberta_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        outputs = roberta_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)
        roberta_predictions.extend(predictions.cpu().numpy())

roberta_predictions = np.array(roberta_predictions)

print("Predictions generated:", len(roberta_predictions))
print("Prediction distribution:", np.unique(roberta_predictions, return_counts=True))

Predictions generated: 1373
Prediction distribution: (array([0, 1, 2]), array([449, 791, 133]))


In [30]:
print("Fine-tuned RoBERTa — Development Set\n")

print("Accuracy:", accuracy_score(dev_labels, roberta_predictions))
print("Macro-F1:", f1_score(dev_labels, roberta_predictions, average="macro"))
print("Weighted-F1:", f1_score(dev_labels, roberta_predictions, average="weighted"))
print("Balanced Accuracy:", balanced_accuracy_score(dev_labels, roberta_predictions))
print("MCC:", matthews_corrcoef(dev_labels, roberta_predictions))

print("\nClassification Report:\n")

print(classification_report(
    dev_labels,
    roberta_predictions,
    labels=[0, 1, 2],
    target_names=["not depression", "moderate", "severe"],
    digits=4,
    zero_division=0
))

Fine-tuned RoBERTa — Development Set

Accuracy: 0.6372906045156591
Macro-F1: 0.59898393748145
Weighted-F1: 0.6344752471401828
Balanced Accuracy: 0.5931907199167575
MCC: 0.36515682560111895

Classification Report:

                precision    recall  f1-score   support

not depression     0.6570    0.5577    0.6033       529
      moderate     0.6448    0.7255    0.6827       703
        severe     0.5263    0.4965    0.5109       141

      accuracy                         0.6373      1373
     macro avg     0.6094    0.5932    0.5990      1373
  weighted avg     0.6373    0.6373    0.6345      1373



In [23]:
distilbert_checkpoint = Path("distilbert_results/checkpoint-2058")  

distilbert_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-uncased"
)

distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    distilbert_checkpoint
)

distilbert_model.eval()

print("DistilBERT loaded from:", distilbert_checkpoint)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBERT loaded from: distilbert_results/checkpoint-2058


In [24]:
distilbert_encodings = distilbert_tokenizer(
    dev_texts,
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)

print("Number of samples:", distilbert_encodings["input_ids"].shape[0])
print("Tokenized shape:", distilbert_encodings["input_ids"].shape)

Number of samples: 1373
Tokenized shape: torch.Size([1373, 256])


In [25]:
distilbert_model.to(device)

distilbert_dataset = TensorDataset(
    distilbert_encodings["input_ids"],
    distilbert_encodings["attention_mask"]
)

distilbert_loader = DataLoader(
    distilbert_dataset,
    batch_size=8,
    shuffle=False
)

distilbert_predictions = []

distilbert_model.eval()

with torch.no_grad():
    for input_ids, attention_mask in distilbert_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        outputs = distilbert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)
        distilbert_predictions.extend(predictions.cpu().numpy())

distilbert_predictions = np.array(distilbert_predictions)

print("Predictions generated:", len(distilbert_predictions))
print(
    "Prediction distribution:",
    np.unique(distilbert_predictions, return_counts=True)
)

Predictions generated: 1373
Prediction distribution: (array([0, 1, 2]), array([486, 763, 124]))


In [26]:
print("Fine-tuned DistilBERT — Development Set\n")

print("Accuracy:", accuracy_score(dev_labels, distilbert_predictions))
print("Macro-F1:", f1_score(dev_labels, distilbert_predictions, average="macro"))
print("Weighted-F1:", f1_score(dev_labels, distilbert_predictions, average="weighted"))
print("Balanced Accuracy:", balanced_accuracy_score(dev_labels, distilbert_predictions))
print("MCC:", matthews_corrcoef(dev_labels, distilbert_predictions))

print("\nClassification Report:\n")

print(classification_report(
    dev_labels,
    distilbert_predictions,
    labels=[0, 1, 2],
    target_names=["not depression", "moderate", "severe"],
    digits=4,
    zero_division=0
))

Fine-tuned DistilBERT — Development Set

Accuracy: 0.6336489439184269
Macro-F1: 0.5942977823995487
Weighted-F1: 0.6315722404613434
Balanced Accuracy: 0.5855997170802304
MCC: 0.3580805832907863

Classification Report:

                precision    recall  f1-score   support

not depression     0.6379    0.5860    0.6108       529
      moderate     0.6474    0.7027    0.6739       703
        severe     0.5323    0.4681    0.4981       141

      accuracy                         0.6336      1373
     macro avg     0.6059    0.5856    0.5943      1373
  weighted avg     0.6319    0.6336    0.6316      1373

